# Customer Churn Prediction Project

You are a Data Analyst working for an e-commerce company that wants to identify customers who are likely to churn.

The objective of this project is to explore customer behavior, prepare the data, train classification models, evaluate their performance, and interpret the business implications of the results.

## Step 0: Import Libraries

Import all required libraries for data analysis, preprocessing, modeling, and evaluation.

- Since this is a classification problem, you will use both Logistic Regression and Decision Tree models to compare their performance using different evaluation metrics.

In [89]:
# Library for data manipulation and analysis
# Write your code here
import pandas as pd

# Splitting data into training and testing sets
# Write your code here
from sklearn.model_selection import train_test_split

# Machine learning models for classification
# Write your code here
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Evaluation metrics to measure model performance
# Write your code here
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score


# Used to scale numerical features to a common range. More details on how to apply it will be provided in a later step.
from sklearn.preprocessing import StandardScaler

---

## Task 1: Data Exploration

Explore the dataset to understand its structure, variables, and target distribution.

- Display the first rows of the dataset using `head()`
- Check data types and missing values using `info()`
- Identify missing values using `isnull().sum()`
- Analyze the target variable using `value_counts()` on `Churn`
- Generate summary statistics using `describe()`
- You are encouraged to explore the dataset further using additional functions

In [90]:
# Load dataset
df = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/ecommerce_churn.csv")

In [91]:
# Display the first rows of the dataset

df.head()

,CustomerID,Churn,Tenure,Gender,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderCount,DaySinceLastOrder
0,54648,0,1.0,Female,4,Single,3,0,2.0,1.0
1,53504,1,3.0,Male,2,Married,9,0,NaN,6.0
2,50722,0,8.0,Male,4,Married,3,0,6.0,4.0
3,50745,0,19.0,Male,4,Divorced,6,0,1.0,3.0
4,53391,1,21.0,Male,3,Single,6,0,2.0,1.0


In [92]:
# Check data types and missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1896 entries, 0 to 1895
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         1896 non-null   int64  
 1   Churn              1896 non-null   int64  
 2   Tenure             1771 non-null   float64
 3   Gender             1896 non-null   object 
 4   SatisfactionScore  1896 non-null   int64  
 5   MaritalStatus      1896 non-null   object 
 6   NumberOfAddress    1896 non-null   int64  
 7   Complain           1896 non-null   int64  
 8   OrderCount         1833 non-null   float64
 9   DaySinceLastOrder  1797 non-null   float64
dtypes: float64(3), int64(5), object(2)
memory usage: 148.2+ KB


In [93]:
# Identify missing values in the dataset
df.isnull().sum()

CustomerID             0
Churn                  0
Tenure               125
Gender                 0
SatisfactionScore      0
MaritalStatus          0
NumberOfAddress        0
Complain               0
OrderCount            63
DaySinceLastOrder     99
dtype: int64

In [94]:
# Analyze the target variable (Churn)
df["Churn"].value_counts(dropna=False)

0    948
1    948
Name: Churn, dtype: int64

In [95]:
# Generate summary statistics for numerical features
df.describe()

,CustomerID,Churn,Tenure,SatisfactionScore,NumberOfAddress,Complain,OrderCount,DaySinceLastOrder
count,1896.000000,1896.000000,1771.000000,1896.000000,1896.000000,1896.000000,1833.000000,1797.000000
mean,52758.544304,0.500000,7.629023,3.194093,4.313819,0.380802,2.905074,4.002782
std,1642.162800,0.500132,8.488319,1.366886,2.671286,0.485712,2.863286,3.653947
min,50001.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,0.000000
25%,51344.500000,0.000000,1.000000,2.000000,2.000000,0.000000,1.000000,1.000000
50%,52751.500000,0.500000,4.000000,3.000000,3.000000,0.000000,2.000000,3.000000
75%,54173.500000,1.000000,13.000000,4.000000,6.000000,1.000000,3.000000,7.000000
max,55630.000000,1.000000,61.000000,5.000000,22.000000,1.000000,16.000000,46.000000


In [96]:
# (Optional) Use additional functions to explore the dataset further

print(df["Gender"].value_counts(dropna=False))
    
print(df["MaritalStatus"].value_counts(dropna=False))

df.nunique()

Male      1146
Female     750
Name: Gender, dtype: int64
Married     873
Single      749
Divorced    274
Name: MaritalStatus, dtype: int64


CustomerID           1896
Churn                   2
Tenure                 33
Gender                  2
SatisfactionScore       5
MaritalStatus           3
NumberOfAddress        14
Complain                2
OrderCount             16
DaySinceLastOrder      21
dtype: int64

---

## Task 2: Data Preprocessing

Prepare the dataset before building the machine learning models.

Perform the following tasks:

- Handle missing values:
  - Remove rows with missing values in the `Churn` column using `dropna`.
  - Apply median imputation to `Tenure`, `SatisfactionScore`, `NumberOfAddress`, `OrderCount`, and `DaySinceLastOrder` using `fillna`.

- Encode categorical variables:
  - Apply one-hot encoding to Gender and MaritalStatus using `pd.get_dummies()`
  - Set `drop_first=True` to avoid redundancy by removing one category from each variable, preventing duplicate information in the dataset.

- Scale numerical features:
  - Standardize `Tenure`, `SatisfactionScore`, `NumberOfAddress`, `OrderCount`, and `DaySinceLastOrder`

- Define the feature set:
  - Create `X` using all predictor variables
  - Exclude `CustomerID` from the feature set

- Define the target variable:
  - Create `y` using the `Churn` column

In [97]:
# Remove missing values in target using dropna()
df = df.dropna(subset=["Tenure", "OrderCount","DaySinceLastOrder"])

# Numerical columns
num_cols = [
    "Tenure", 
    "SatisfactionScore", 
    "NumberOfAddress", 
    "OrderCount", 
    "DaySinceLastOrder"     
]

# Apply median imputation to each numerical column using fillna()
for col in num_cols:
    df[col]=df[col].fillna(df[col].median())

In [98]:
# Identify categorical variables

categorical_cols = (["Gender", "MaritalStatus"])

# Apply One-Hot Encoding using pd.get_dummies()
df_encoded = pd.get_dummies(df, categorical_cols,
drop_first=True)
df_encoded

,CustomerID,Churn,Tenure,SatisfactionScore,NumberOfAddress,Complain,OrderCount,DaySinceLastOrder,Gender_Male,MaritalStatus_Married,MaritalStatus_Single
0,54648,0,1.0,4,3,0,2.0,1.0,0,0,1
2,50722,0,8.0,4,3,0,6.0,4.0,1,1,0
3,50745,0,19.0,4,6,0,1.0,3.0,1,0,0
4,53391,1,21.0,3,6,0,2.0,1.0,1,0,1
5,50515,0,22.0,5,3,0,1.0,2.0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
1891,51413,0,18.0,5,3,0,1.0,0.0,1,1,0
1892,55179,0,10.0,1,3,0,2.0,9.0,1,0,1
1893,55053,1,1.0,4,3,1,2.0,1.0,0,0,1
1894,53103,0,20.0,3,4,0,2.0,9.0,0,1,0


In [99]:
# Define Features by removing the target variable and the customer identifier
X = df_encoded.drop(["Churn", "CustomerID"], axis=1)

# Define Target
y = df["Churn"]

print(X.shape)
print(y.shape)

(1609, 9)
(1609,)


## Task 3: Train-Test Split

Split the dataset into training and testing subsets.

Perform the following tasks:

- Ensure `train_test_split` is imported in Step 0 before splitting the dataset
- Split the data into training and testing sets
- Use `test_size=0.20`
- Use `random_state=42` to ensure reproducible results
- Display the shapes of `X_train`, `X_test`, `y_train`, and `y_test`
- Apply feature scaling to the training and testing sets using `StandardScaler`. Refer to the **comments** within the code for guidance.


In [100]:
# Split the dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Display dataset shapes

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (1287, 9)
X_test shape: (322, 9)
y_train shape: (1287,)
y_test shape: (322,)


### Feature Scaling

In this step, you will apply feature scaling to the dataset. As you saw in previous lessons, numerical variables should be on the same scale when using many machine learning algorithms. This is important because features with larger values can dominate the model and affect its performance.

Unlike some basic models where scaling is less critical, in classification algorithms such as Logistic Regression and Decision Trees, scaling helps ensure that the model learns patterns more effectively from all features.

To perform this, execute the following lines of code:

In [101]:
# Scale numerical features
scaler = StandardScaler()

# Create copies to avoid modifying original data directly
X_train = X_train.copy()
X_test = X_test.copy()

# Learn scaling parameters from the training data and apply the transformation
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# Apply the same transformation to test data
X_test[num_cols] = scaler.transform(X_test[num_cols])

---

## Task 4: Train Classification Models

Train classification models using the training dataset.

Perform the following tasks:

**Logistic Regression**

- Create a Logistic Regression model.
- Set `max_iter=1000` to allow the model enough iterations to find optimal parameters
- Fit the model using `X_train` and `y_train`
- Generate predictions using `X_test`

**Decision Tree**

- Create a Decision Tree Classifier
- Set `random_state=42` to ensure reproducible results
- Experiment with different values of `max_depth` (for example 3, 4, and 5) and analyze how model performance changes. Select the value that provides the best result.
- Fit the model using `X_train` and `y_train`
- Generate predictions using `X_test`

In [102]:
# Logistic Regression

# Initialize Logistic Regression model (use max_iter)
log_model = LogisticRegression(random_state=42, max_iter=1000)

# Train the model using the training data
log_model.fit(X_train, y_train)

# Generate predictions on the test set
y_pred_log = log_model.predict(X_test)

In [103]:
# Decision Tree Classifier

# Initialize Decision Tree Classifier: use max_depth to control the complexity of the tree (helps prevent overfitting) and
# random_state ensures reproducible results

tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
#max_depth=3 provides the best results out of the range 3-5

# Train the model using the training data
tree_model.fit(X_train, y_train)

# Generate predictions on the test set
y_pred_tree = tree_model.predict(X_test)

---

## Task 5: Model Evaluation

Evaluate the performance of both models.

Perform the following tasks:

- Calculate Accuracy using the test set predictions.
- Calculate Precision using the test set predictions.
- Calculate Recall using the test set predictions (Use Recall as the primary metric when selecting the best model for customer churn prediction.)
- Generate a Confusion Matrix
- Compare the results of both models
- Compare training and test performance for the models to analyze potential overfitting or underfitting by executing the cells below.

There is no strict rule to define overfitting, but the following guideline can help interpret the difference between training and test performance:

| Difference | Interpretation |
|------------|----------------|
| < 0.02 | Small difference (likely noise) |
| 0.02 – 0.05 | Moderate difference (may be relevant) |
| > 0.05 | Large difference (possible overfitting) |

- If the difference is **small**, the model is well generalized. Prefer that model, especially if it also has higher Recall.

In [104]:
# Logistic Regression Evaluation

print("Logistic Regression Performance")
print()

print("Accuracy:", accuracy_score(y_test, y_pred_log))

print("Precision:", precision_score(y_test, y_pred_log))

print("Recall:", recall_score(y_test, y_pred_log))

print()
print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred_log))

Logistic Regression Performance

Accuracy: 0.8229813664596274
Precision: 0.7987804878048781
Recall: 0.8451612903225807

Confusion Matrix
[[134  33]
 [ 24 131]]


### Overfitting Check (Train vs Test Performance - Logistic Regression)

No additional code is required. Execute the cells below and interpret the results.

In [105]:
# Logistic Regression comparison
# Compare training vs test accuracy to check for overfitting or underfitting
# The .score() method returns accuracy for classification models

train_acc_log = log_model.score(X_train, y_train)
test_acc_log = log_model.score(X_test, y_test)

print("Logistic Regression Performance Comparison")
print("-------------------------------")

# Accuracy is used here to check how well the model performs on seen vs unseen data
print("Train Accuracy:", train_acc_log)
print("Test Accuracy:", test_acc_log)
print()

print("Difference (Train - Test):", train_acc_log - test_acc_log)

Logistic Regression Performance Comparison
-------------------------------
Train Accuracy: 0.8135198135198135
Test Accuracy: 0.8229813664596274

Difference (Train - Test): -0.009461552939813878


In [106]:
# Decision Tree Evaluation

print("Decision Tree Performance")
print()

print("Accuracy:", accuracy_score(y_test, y_pred_tree))

print("Precision:", precision_score(y_test, y_pred_tree))

print("Recall:", recall_score(y_test, y_pred_tree))

print()
print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred_tree))

Decision Tree Performance

Accuracy: 0.8043478260869565
Precision: 0.8108108108108109
Recall: 0.7741935483870968

Confusion Matrix
[[139  28]
 [ 35 120]]


### Overfitting Check (Train vs Test Performance - Decision Tree)
No additional code is required. Execute the cells below and interpret the results.

In [107]:
# Train vs Test performance - Decision Tree

train_accuracy = tree_model.score(X_train, y_train)
test_accuracy = tree_model.score(X_test, y_test)

print("Decision Tree Performance Comparison")
print("-------------------------------")
print("Train Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)
print()

print("Difference (Train - Test):", train_accuracy - test_accuracy)

Decision Tree Performance Comparison
-------------------------------
Train Accuracy: 0.8034188034188035
Test Accuracy: 0.8043478260869565

Difference (Train - Test): -0.00092902266815309


### Compare model performance
- Compare the performance of both models using Accuracy, Precision, and Recall  
- Select the best model based on the highest Recall, since in customer churn prediction it is more important to correctly identify customers who are likely to leave the company than to minimize other types of errors 

In [108]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_log),
        accuracy_score(y_test, y_pred_tree)
    ],
    "Precision": [
        precision_score(y_test, y_pred_log),
        precision_score(y_test, y_pred_tree)
    ],
    "Recall": [
        recall_score(y_test, y_pred_log),
        recall_score(y_test, y_pred_tree)
    ]
})

comparison

,Model,Accuracy,Precision,Recall
0,Logistic Regression,0.822981,0.798780,0.845161
1,Decision Tree,0.804348,0.810811,0.774194


---

## Conclusion

Replace the placeholders below:

The Logistic Regression model achieved the highest Recall and was selected as the best model for this project.

- Accuracy: 0.822981
- Precision: 0.798780
- Recall: 0.845161

Recall was considered the most important metric because the main objective of churn prediction is to identify customers who are likely to leave the company.

Logistic Regression had a higher Recall and Accuracy Score, although the model's Precision Score was lower. It also had a slightly higher difference between training and testing performance, but both models had low differences (<0.01). Due to Customer Churn being the target, it is most important to correctly identify customers who are likely to leave the company, so Recall Score was prioritized over other classification metrics, as it measures how many of the actual positive cases the model successfully identified. 

The model found that 48% of the customer dataset have churned. Recommend conducting further research into what caused the uptick in churn, such as poor customer service or product price increases.

Overall, the model performs well and demonstrates that customer characteristics such as tenure, satisfaction score, and days since last order are useful for predicting churn.